In [273]:
from tqdm.auto import tqdm
tqdm.pandas()

import os
import sys

module_path = os.path.abspath(os.path.join('..'))
if not module_path in sys.path:
    sys.path.insert(0, module_path)

from innoprod.sheet_tools import get_sheet_dfs
from innoprod.wrangling.wrangling_tools import is_non_empty
from innoprod.wrangling.msyh_data_sharing import wrangle_roadmaps
from innoprod.text_analysis.code_analysis import CodeAnalysis

In [274]:
data = get_sheet_dfs()
roadmaps_df = wrangle_roadmaps(data['Roadmaps'])

In [275]:
cols = [
    'Summary review of Edge Digital diagnostic report & current state and key improvement areas',
    'What are the internal barriers to growth? How do you intend to finance future growth? Are there sufficient leadership and management skills in the business to achieve your growth? What opportunities do you have to expand into new markets?',
    'Level of current Strategic Digital Skills/knowledge in the business',
    'Level of current Technical Digital Skills/knowledge in the business',
    'Whether the business is already investing/adopting/utilising Industry 4.0 Technologies, with examples',
    'Summary of the identified problems, including Gap Analysis',
    'Key potential industry 4.0 solutions to address the identified problems/gaps',
    'Recommended Action Plan to utilise Industry 4.0 Technology',
    'Overview of qualitative benefits of recommended Action Plan (positive/negative)',
    'Skills and other requirements that will be needed to successfully implement the recommended Action Plan',
    'What has been your overall opinion of the support you have received in this programme? (Add comments)'
]

# All question responses as long table

In [276]:
responses_df = roadmaps_df[['Client ID'] + cols].melt(id_vars=['Client ID'], value_vars=cols, var_name='Question', value_name='Response')
responses_df = responses_df[is_non_empty(responses_df['Response'])]

responses_df

,Client ID,Question,Response
0,f3fff05d-1a28-e8f3-c5f6-670d1d0797e3,Summary review of Edge Digital diagnostic repo...,[REDACTED] business has completed an edge digi...
1,458079bc-a5ab-2055-d514-6733331e9c5f,Summary review of Edge Digital diagnostic repo...,[REDACTED] Score: 7 STATUS: Based on your resp...
2,0369B4F9-9E83-E83D-9E6B-BF82E264A2BA,Summary review of Edge Digital diagnostic repo...,The EDD has identified that the Company has in...
3,e9b5b5a2-1ba0-1d3a-a374-67ed061c1e40,Summary review of Edge Digital diagnostic repo...,Summary review of [REDACTED] diagnostic report...
4,052CB881-3557-DCFA-E472-0E55A5D04590,Summary review of Edge Digital diagnostic repo...,The business has rated itself at level 4 in te...
...,...,...,...
2414,3044E4BE-D41D-1AD3-7DFE-D22F7E559972,What has been your overall opinion of the supp...,No negatives to report. Very good process and ...
2415,0EC71AFB-5867-DE5F-5061-32AEAE4B24BF,What has been your overall opinion of the supp...,Good. The only thing to say is it is difficult...
2416,2346FC83-5B42-B90D-3BBB-16BFD156902E,What has been your overall opinion of the supp...,"as usual a simple process, managed for us by o..."
2417,2AA6320B-75FF-BABC-1E79-EB96B0AE650E,What has been your overall opinion of the supp...,excellent support as always. I had received su...


# Break into sentences

In [277]:
from nltk.tokenize import PunktSentenceTokenizer
tokenizer = PunktSentenceTokenizer()

sentences_df = responses_df.copy()
sentences_df['Sentences'] = sentences_df.apply(lambda row: tokenizer.tokenize(row['Response']), axis=1)
sentences_df = sentences_df[['Client ID', 'Question', 'Sentences']].explode('Sentences').reset_index().rename(columns={'index' : 'Sentence Number', 'Sentences' : 'Sentence'})
sentences_df['Sentence Number'] = sentences_df.groupby('Sentence Number').cumcount() + 1
sentences_df

,Sentence Number,Client ID,Question,Sentence
0,1,f3fff05d-1a28-e8f3-c5f6-670d1d0797e3,Summary review of Edge Digital diagnostic repo...,[REDACTED] business has completed an edge digi...
1,2,f3fff05d-1a28-e8f3-c5f6-670d1d0797e3,Summary review of Edge Digital diagnostic repo...,They are aware of [REDACTED] weaknesses in not...
2,3,f3fff05d-1a28-e8f3-c5f6-670d1d0797e3,Summary review of Edge Digital diagnostic repo...,[REDACTED] company has invested in appropriate...
3,4,f3fff05d-1a28-e8f3-c5f6-670d1d0797e3,Summary review of Edge Digital diagnostic repo...,They also recognise that they struggle to util...
4,5,f3fff05d-1a28-e8f3-c5f6-670d1d0797e3,Summary review of Edge Digital diagnostic repo...,[REDACTED] key improvement area at this time i...
...,...,...,...,...
8027,2,0EC71AFB-5867-DE5F-5061-32AEAE4B24BF,What has been your overall opinion of the supp...,The only thing to say is it is difficult somet...
8028,1,2346FC83-5B42-B90D-3BBB-16BFD156902E,What has been your overall opinion of the supp...,"as usual a simple process, managed for us by o..."
8029,1,2AA6320B-75FF-BABC-1E79-EB96B0AE650E,What has been your overall opinion of the supp...,excellent support as always.
8030,2,2AA6320B-75FF-BABC-1E79-EB96B0AE650E,What has been your overall opinion of the supp...,I had received support from Marcus and Oxford ...


# Pre-process sentences

In [278]:
sentences_df['Cleaned Sentence'] = sentences_df['Sentence'].str.lower()

# TODO Standardise acronyms? 
# Need to be very careful if removing stop words, as the standard stop word lists contains words like "not" that are important here.
# Consider lemmatisation/stemming?

# Count repeated sentences

In [279]:
recurring_sentences_df = sentences_df['Cleaned Sentence'].value_counts().to_frame().reset_index().rename(columns={'count': 'Count'})
# recurring_sentences_df = recurring_sentences_df.merge(sentences_df[['Cleaned Sentence', 'Question']], on='Cleaned Sentence', how='left').drop_duplicates(subset=['Cleaned Sentence', 'Question'])
# recurring_sentences_df = recurring_sentences_df.groupby(['Cleaned Sentence', 'Count'])['Question'].apply(list).to_frame().rename(columns={'Question': 'Questions'}).reset_index().sort_values(by='Count', ascending=False)
# recurring_sentences_df['Num Questions'] = recurring_sentences_df['Questions'].apply(len)
recurring_sentences_df

,Cleaned Sentence,Count
0,the development of a formal digital strategy s...,78
1,impact of the implementation of the action pla...,59
2,as with many business one of the main barriers...,55
3,this will require investment in specialist equ...,52
4,this will be underpinned by increased function...,50
...,...,...
4550,the only thing to say is it is difficult somet...,1
4551,"as usual a simple process, managed for us by o...",1
4552,excellent support as always.,1
4553,i had received support from marcus and oxford ...,1


# Manual iterative pattern matching

In [280]:
code_analysis = CodeAnalysis(recurring_sentences_df)

### Sentence 1

In [281]:
s = code_analysis.get_next_sentence()
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [282]:
code_analysis.add_keyword('digital strategy', 'Digital Strategy')
code_analysis.add_keyword('project plans to support implementation', 'Implementation Project Plan(s)')
code_analysis.add_quantifier('development of a formal KEYWORD should be the starting point', 'Absence')
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [283]:
code_analysis.mark_sentence_analysed(s)

### Sentence 2

In [284]:
s = code_analysis.get_next_sentence()
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [285]:
code_analysis.add_keyword('implementation of the action plan', 'Implementation Project Plan(s)')
code_analysis.add_quantifier('impact of the KEYWORD will be', 'Expected benefit')
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [286]:
code_analysis.mark_sentence_analysed(s)

### Sentence 3

In [287]:
s = code_analysis.get_next_sentence()
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [288]:
code_analysis.add_keyword('finance to support investment', 'Capital Investment')
code_analysis.add_quantifier(r'one of the main barriers to investment in \[redacted\] has been accessing the KEYWORD', 'Barrier/Constraint')
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [289]:
code_analysis.mark_sentence_analysed(s)

### Sentence 4

In [290]:
s = code_analysis.get_next_sentence()
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [291]:
code_analysis.add_keyword('investment in specialist equipment', 'Capital Investment')
code_analysis.add_quantifier(r'this will require KEYWORD', 'Need for')
code_analysis.add_keyword('data collection and tracking', 'Use of Data')
code_analysis.add_quantifier(r'for future KEYWORD', 'Future use')
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [292]:
code_analysis.mark_sentence_analysed(s)

### Sentence 5

In [293]:
s = code_analysis.get_next_sentence()
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [294]:
code_analysis.add_keyword('data collection and analysis', 'Use of Data')
code_analysis.add_quantifier('to allow greater use of KEYWORD', 'Future use')
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [295]:
code_analysis.mark_sentence_analysed(s)

### Sentence 6

In [296]:
s = code_analysis.get_next_sentence()
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [297]:
code_analysis.add_keyword('data collection and management', 'Use of Data')
code_analysis.add_quantifier('it is expected that this increased functionality within the KEYWORD area', 'Future use')
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [298]:
code_analysis.mark_sentence_analysed(s)

### Sentence 7

In [299]:
s = code_analysis.get_next_sentence()
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [300]:
code_analysis.add_keyword('data collection', 'Use of Data')
code_analysis.add_quantifier('the potential move to increasing productivity will require increased KEYWORD', 'Future use')
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [301]:
code_analysis.mark_sentence_analysed(s)

### Sentence 8

In [302]:
s = code_analysis.get_next_sentence()
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [303]:
code_analysis.add_keyword('skills gaps', 'Skills/Training')
code_analysis.add_keyword('training and development', 'Skills/Training')
code_analysis.add_quantifier('need to identify key KEYWORD', 'Future use')
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [304]:
code_analysis.mark_sentence_analysed(s)

### Sentence 9

In [305]:
s = code_analysis.get_next_sentence()
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [306]:
code_analysis.add_keyword('new technology', 'Innovation Adoption')
code_analysis.add_quantifier('implementation of KEYWORD is seen as a key driver', 'Attitude: positive')
code_analysis.add_keyword('internal skills', 'Skills/Training')
code_analysis.add_quantifier('the company are committed to developing KEYWORD', 'Attitude: positive')
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [307]:
code_analysis.mark_sentence_analysed(s)

### Sentence 10

In [308]:
s = code_analysis.get_next_sentence()
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [309]:
code_analysis.add_keyword('technical skills', 'Skills/Training')
code_analysis.add_quantifier('levels of KEYWORD in the business are largely limited', 'Presence: limited')
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [310]:
code_analysis.mark_sentence_analysed(s)

### Sentence 11

In [311]:
s = code_analysis.get_next_sentence()
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [312]:
code_analysis.add_keyword('sales order volume growth', 'Growth')
code_analysis.add_quantifier('the expectation is that this could drive KEYWORD', 'Expected benefit')
code_analysis.add_keyword('better understanding potential for new product/service developments', 'Offering innovation')
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [313]:
code_analysis.mark_sentence_analysed(s)

### Sentence 12

In [314]:
s = code_analysis.get_next_sentence()
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [315]:
code_analysis.add_keyword('an overall improvement in productivity', 'Productivity')
code_analysis.add_quantifier('the main qualititive impact that the company is pursuing through implementing increased digitisation is KEYWORD', 'Expected benefit')
code_analysis.add_keyword('time/capacity', 'Internal Resources')
code_analysis.add_quantifier('they are KEYWORD constrained', 'Barrier/Constraint')
code_analysis.add_keyword('business volumes', 'Growth')
code_analysis.add_quantifier('KEYWORD are increasing', 'Currently increasing')
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [316]:
code_analysis.mark_sentence_analysed(s)

### Sentence 13

In [317]:
s = code_analysis.get_next_sentence()
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [318]:
code_analysis.add_keyword(r'\[redacted\] management', 'Leadership')
code_analysis.add_quantifier('KEYWORD are progressive and forward thinking', 'Attitude: positive')
code_analysis.add_keyword('technology', 'Innovation Adoption')
code_analysis.add_quantifier('see KEYWORD as a means of enhancing productivity', 'Attitude: positive')
code_analysis.add_keyword('investment', 'Capital Investment')
code_analysis.add_quantifier('need to balance KEYWORD with', 'Risk cautious')
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [319]:
code_analysis.mark_sentence_analysed(s)

### Sentence 14

In [320]:
code_analysis.quantifiers

{'development of a formal KEYWORD should be the starting point': 'Absence',
 'impact of the KEYWORD will be': 'Expected benefit',
 'one of the main barriers to investment in \\[redacted\\] has been accessing the KEYWORD': 'Barrier/Constraint',
 'this will require KEYWORD': 'Need for',
 'for future KEYWORD': 'Future use',
 'to allow greater use of KEYWORD': 'Future use',
 'it is expected that this increased functionality within the KEYWORD area': 'Future use',
 'the potential move to increasing productivity will require increased KEYWORD': 'Future use',
 'need to identify key KEYWORD': 'Future use',
 'implementation of KEYWORD is seen as a key driver': 'Attitude: positive',
 'the company are committed to developing KEYWORD': 'Attitude: positive',
 'levels of KEYWORD in the business are largely limited': 'Presence: limited',
 'the expectation is that this could drive KEYWORD': 'Expected benefit',
 'the main qualititive impact that the company is pursuing through implementing increased di

In [321]:
s = code_analysis.get_next_sentence()
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [322]:
# code_analysis.add_keyword('client data management system', 'Use of data')
# code_analysis.add_quantifier('KEYWORD which will provide capacity', 'Expected benefit')
# code_analysis.add_keyword('', '')
# code_analysis.add_quantifier('KEYWORD', '')
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [323]:
code_analysis.mark_sentence_analysed(s)

### Sentence 15

In [324]:
s = code_analysis.get_next_sentence()
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [325]:
code_analysis.add_keyword('leadership team', 'Leadership')
code_analysis.add_quantifier('business has a strong and experienced KEYWORD', 'Presence: strongly positive')
code_analysis.add_keyword('strategic management experience', 'Strategic skills')
code_analysis.add_quantifier('with extensive KEYWORD', 'Presence: strongly positive')
code_analysis.add_keyword('use of digital solutions', 'Innovation Adoption')
code_analysis.add_quantifier('this extends to understanding the benefits of enhancing the KEYWORD', 'Attitude: positive')
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [326]:
code_analysis.mark_sentence_analysed(s)

### Sentence 16

In [327]:
s = code_analysis.get_next_sentence()
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [328]:
code_analysis.add_keyword('systems or processes to collate data', 'Use of data')
code_analysis.add_quantifier('currently has little in the way of KEYWORD', 'Presence: weak')
code_analysis.add_keyword('low level machine data capture', 'Use of data')
code_analysis.add_quantifier('this could be readily addressed through the implementation of KEYWORD', 'Opportunity identified')
code_analysis.add_keyword('indicative levels of machine utilisation/laten capacity', 'Internal Resources')
code_analysis.add_quantifier('using sensor technology and associated reporting to find KEYWORD', 'Expected benefit')
# TODO fix clash between 'technology' as a keyword vs as part of a quantifier
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [329]:
code_analysis.mark_sentence_analysed(s)

### Sentence 17

In [330]:
s = code_analysis.get_next_sentence()
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [331]:
code_analysis.add_keyword('skills development', 'Skills/Training')
code_analysis.add_quantifier('KEYWORD will be a key requirement', 'Presence: insufficient')
code_analysis.add_keyword('adoption of digital technologies', 'Innovation Adoption')
code_analysis.add_keyword('business growth objectives', 'Growth')
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [332]:
code_analysis.mark_sentence_analysed(s)

### Sentence 18

In [333]:
s = code_analysis.get_next_sentence()
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [334]:
code_analysis.add_keyword('opportunities for increased digitisation/automation', 'Innovation Adoption')
code_analysis.add_quantifier('staff are actively encouraged to identify and recommend KEYWORD', 'Attitude: positive')
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [335]:
code_analysis.mark_sentence_analysed(s)

### Sentence 19

In [336]:
s = code_analysis.get_next_sentence()
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [337]:
code_analysis.mark_sentence_analysed(s)

### Sentence 20

In [338]:
s = code_analysis.get_next_sentence()
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [339]:
# TODO add keywords & quantifiers around identified opportunities?
# code_analysis.add_keyword('', '')
# code_analysis.add_quantifier('KEYWORD', '')
# code_analysis.display_sentence_with_highlights(s)

In [340]:
code_analysis.mark_sentence_analysed(s)

### Sentence 21

In [341]:
s = code_analysis.get_next_sentence()
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [342]:
# code_analysis.add_keyword('', '')
# code_analysis.add_quantifier('KEYWORD', '')
# code_analysis.display_sentence_with_highlights(s)

In [343]:
code_analysis.mark_sentence_analysed(s)

### Sentence 22

In [344]:
s = code_analysis.get_next_sentence()
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [345]:
# code_analysis.add_keyword('', '')
# code_analysis.add_quantifier('company should consider support in the development and implementation of a KEYWORD', '')
# code_analysis.display_sentence_with_highlights(s)

In [346]:
code_analysis.mark_sentence_analysed(s)

### Sentence 22

In [347]:
s = code_analysis.get_next_sentence()
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [348]:
# code_analysis.add_keyword('', '')
code_analysis.add_quantifier('attend ms leading digital transformation course to develop formal KEYWORD', 'Presence: missing')
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [349]:
code_analysis.mark_sentence_analysed(s)

### Sentence 23

In [350]:
s = code_analysis.get_next_sentence()
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [351]:
# code_analysis.add_keyword('', '')
# code_analysis.add_quantifier('KEYWORD', '')
# code_analysis.display_sentence_with_highlights(s)

In [352]:
code_analysis.mark_sentence_analysed(s)

### Sentence 24

In [353]:
s = code_analysis.get_next_sentence()
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [354]:
# TODO fix clash between 'investment' as a keyword vs as part of a quantifier
code_analysis.mark_sentence_analysed(s)

### Sentence 25

In [355]:
s = code_analysis.get_next_sentence()
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [356]:
# code_analysis.add_keyword('', '')
# code_analysis.add_quantifier('KEYWORD', '')
# code_analysis.display_sentence_with_highlights(s)

In [357]:
code_analysis.mark_sentence_analysed(s)

### Sentence 26

In [358]:
s = code_analysis.get_next_sentence()
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [359]:
code_analysis.add_keyword('digital leadership skills', 'Leadership skills')
code_analysis.add_quantifier('development of KEYWORD should also be seen as essential', 'Presence: missing')
code_analysis.add_quantifier('which would support the creation and implementation of a KEYWORD', 'Presence: missing')
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [360]:
code_analysis.mark_sentence_analysed(s)

### Sentence 27

In [361]:
s = code_analysis.get_next_sentence()
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [362]:
# code_analysis.add_keyword('', '')
# code_analysis.add_quantifier('KEYWORD', '')
# code_analysis.display_sentence_with_highlights(s)

In [363]:
code_analysis.mark_sentence_analysed(s)

### Sentence 28

In [364]:
s = code_analysis.get_next_sentence()
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [365]:
code_analysis.mark_sentence_analysed(s)

### Sentence 29

In [366]:
s = code_analysis.get_next_sentence()
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [367]:
# code_analysis.add_keyword('', '')
# code_analysis.add_quantifier('KEYWORD', '')
# code_analysis.display_sentence_with_highlights(s)

In [368]:
code_analysis.mark_sentence_analysed(s)

### Sentence 30

In [369]:
s = code_analysis.get_next_sentence()
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [370]:
code_analysis.add_keyword('digital adoption strategy', 'Digital Strategy')
code_analysis.add_quantifier('will require the development of a clear KEYWORD', 'Presence: missing')
code_analysis.display_sentence_with_highlights(s)

HTML(value='<style>.tooltip {position:relative;display:inline-block;cursor:pointer;} .tooltiptext{visibility:h…

In [371]:
code_analysis.mark_sentence_analysed(s)

In [372]:
code_analysis.recurring_sentences_df[code_analysis.recurring_sentences_df['Analysed'] == False]

,Cleaned Sentence,Count,Analysed
31,in addition to specialist support to develop a...,21,False
32,"reduce time on jobs and processes, streamline ...",21,False
33,the main solution to address the company's pro...,21,False
34,it will also enable a greater understanding of...,21,False
35,the lack of a formal digital strategy demonstr...,20,False
...,...,...,...
4550,the only thing to say is it is difficult somet...,1,False
4551,"as usual a simple process, managed for us by o...",1,False
4552,excellent support as always.,1,False
4553,i had received support from marcus and oxford ...,1,False
